# YaRN Rotary Position Embedding
*Extending RoPE to longer contexts via smooth frequency interpolation — the positional encoding in DeepSeek-V4-DSpark.*

---
**Component:** `learning/kernels/yarn/`
**Companion kernels:** `v0_naive.py` → `sm89_v1_yarn_freqs.py`
**Source:** `model.py::precompute_freqs_cis` with `original_seq_len > 0`


## Abstract

*Imagine teaching someone to read a clock by only ever showing them the hours 1 through 6. They'd learn perfectly within that range — but ask them to tell time on a full 12-hour clock and they'd struggle. Language models face exactly this problem when asked to handle sequences longer than they trained on: the internal "clock" that tracks position goes into unfamiliar territory. YaRN fixes this by gently stretching that clock — slowing down the rotation frequencies that are responsible for long-range position while leaving short-range frequencies completely untouched. The result is a model that gracefully handles 40× longer sequences without any additional fine-tuning.*

---

## What Is YaRN?

Every transformer needs to answer one fundamental question before it can do anything useful: *where is each token in the sequence?* Without position information, "the cat sat on the mat" and "mat sat the on cat the" look completely identical — just a bag of words. Position matters.

**How RoPE encodes position.** Rather than adding a separate position vector to each token, Rotary Position Embedding (RoPE) *rotates* the query and key vectors by an angle that depends on the token's position. Here's the clever part: when you compute the dot product Q · K later, the rotations naturally cancel out so that only the *relative distance* between the two tokens survives in the score. Think of two compass needles both rotated by the same amount — their relative direction is unchanged regardless of which absolute direction they're pointing.

**The frequency ladder.** RoPE doesn't use a single rotation speed for all dimensions. Instead, it assigns a different frequency to each pair of dimensions:
- **High-frequency pairs** (small index $i$) spin fast, completing a full rotation every few tokens — they track nearby words.
- **Low-frequency pairs** (large index $i$) spin slowly, taking thousands of tokens to complete one rotation — they track long-range structure.

**What goes wrong beyond the training length.** During training the model only ever sees positions up to $L_{\text{train}}$, so all rotation angles stay in a comfortable, well-learned range. Extend the sequence to $L_{\text{test}} \gg L_{\text{train}}$ and the medium-frequency pairs suddenly visit angle combinations the model has never encountered. Attention scores become unpredictable and quality collapses.

**YaRN's fix.** YaRN identifies exactly which frequency pairs are in trouble and slows them down by just enough so that even the longest target sequences stay within the range of angles the model saw during training. It does this with a smooth transition — no sudden jumps — so the model never sees a discontinuity in how positions are encoded.

In [ ]:
import math
print("setup complete")


## Review: Baseline RoPE Frequencies

Before we can stretch the frequencies, we need to understand what we're stretching. RoPE assigns a base rotation rate to each pair of dimensions following a geometric sequence:

$$\theta_i = \text{base}^{-2i/d_{\text{rope}}}, \qquad i = 0, 1, \ldots, \tfrac{d_{\text{rope}}}{2} - 1$$

**In plain English:** Pair 0 gets frequency 1 (rotates the fastest). Pair 1 gets frequency $\text{base}^{-2/d}$ (a bit slower). This continues all the way to the last pair, which rotates at frequency $\text{base}^{-1}$ (the slowest). With `base = 10000` and `d = 64`, the slowest pair completes one full rotation every 10,000 tokens — hence the name "10K base RoPE."

> **Note:** DSpark uses `rope_head_dim = 64`, giving 32 frequency pairs per head. The remaining `nope_head_dim = head_dim - rope_head_dim` dimensions don't participate in RoPE at all — they carry content information without any positional rotation.

In [ ]:
base = 10000.0
d = 64  # rope_head_dim in DSpark
n_pairs = d // 2  # 32 frequency pairs

# Baseline RoPE frequencies
freqs_base = [1.0 / (base ** (2 * i / d)) for i in range(n_pairs)]

print("Baseline RoPE frequencies (first and last few):")
print(f"{'pair i':>8} {'theta_i':>12} {'wavelength 2pi/theta':>22}")
for i in [0, 4, 8, 16, 24, 28, 31]:
    theta = freqs_base[i]
    wavelength = 2 * math.pi / theta
    print(f"{i:8d} {theta:12.6f} {wavelength:22.1f}")


## The Problem: Why Long Sequences Break RoPE

Here's the core issue. At position $t$, dimension pair $i$ applies a rotation of $t \cdot \theta_i$ radians. For short sequences, every angle stays within a range the model has learned to interpret. But extend the sequence far beyond the training length and things go wrong for the *medium-frequency* pairs.

Let's make this concrete. Suppose the training length is $L = 4096$ and we're extending to $L' = 128{,}000$ (32× longer). Consider a pair with $\theta_i = 0.01$:

| Position | Angle | Seen during training? |
|----------|-------|----------------------|
| $t = 4096$ | $40.96$ rad | Yes (boundary of training) |
| $t = 128{,}000$ | $1280$ rad | No — 31× beyond training |

The pair completes $1280 / (2\pi) \approx 204$ full cycles at the new length, but only $6.5$ during training. The model has no intuition for attention patterns at this cycle count.

**Which pairs are vulnerable?** Counterintuitively, the very *high*-frequency pairs are fine — they spin so fast that even within the training window they've completed many cycles, so the model has seen every possible angle. The very *low*-frequency pairs barely move at all, so extrapolation barely changes their angle. The danger zone is the middle: pairs whose wavelength is comparable to the training context length. YaRN focuses all its correction energy precisely on this range.

## YaRN Step 1: Finding Which Frequencies Need Help

YaRN asks a precise diagnostic question for each frequency pair: *how many complete rotation cycles does this pair see within the training window?* That count tells us how well the model knows this frequency:

$$\text{cycles in training} = \frac{L_{\text{train}} \cdot \theta_i}{2\pi}$$

- **Very few cycles** (less than $\beta_{\text{slow}} / 2\pi$) → the pair is effectively "DC" (barely moves). Apply full scaling — divide the frequency by the extension factor.
- **Many cycles** (more than $\beta_{\text{fast}} / 2\pi$) → the pair is well-trained at all angles. Leave it untouched.
- **In between** → blend smoothly using a linear ramp.

The function `find_correction_dim(num_rotations, ...)` solves the inverse problem: given a target number of cycles, it finds the dimension boundary where that crossover happens. Think of it as the "tuning dial" that sets where the correction kicks in and where it fades out.

In [ ]:
def find_correction_dim(num_rotations, dim, base, max_seq_len):
    """Pair index where RoPE makes exactly num_rotations full turns over max_seq_len positions."""
    return dim * math.log(max_seq_len / (num_rotations * 2 * math.pi)) / (2 * math.log(base))

def find_correction_range(low_rot, high_rot, dim, base, max_seq_len):
    low = math.floor(find_correction_dim(low_rot, dim, base, max_seq_len))
    high = math.ceil(find_correction_dim(high_rot, dim, base, max_seq_len))
    return max(low, 0), min(high, dim - 1)

# DSpark config values
beta_fast = 32
beta_slow = 1
original_seq_len = 4096  # example training length
d = 64
base = 10000.0

low, high = find_correction_range(beta_fast, beta_slow, d, base, original_seq_len)
print(f"Training length: {original_seq_len}")
print(f"beta_fast={beta_fast} → correction_dim = {find_correction_dim(beta_fast, d, base, original_seq_len):.2f} → low = {low}")
print(f"beta_slow={beta_slow} → correction_dim = {find_correction_dim(beta_slow, d, base, original_seq_len):.2f} → high = {high}")
print(f"\nResult: correction range = [{low}, {high}]")
print(f"  pairs 0..{low-1}: >32 rotations over training length → no scaling")
print(f"  pairs {low}..{high}: 1-32 rotations → smooth blend")
print(f"  pairs {high+1}..{d//2 - 1}: <1 rotation → full /factor scaling")


## YaRN Step 2: A Smooth Transition Between Corrected and Uncorrected

Once we know the two boundary dimensions, we need a smooth way to blend between "fully corrected" and "completely untouched." YaRN uses a linear ramp — the simplest possible transition:

$$\text{smooth}(i) = 1 - \text{clamp}\!\left(\frac{i - \text{low}}{\text{high} - \text{low}},\ 0,\ 1\right)$$

Picture a lighting dimmer switch: below `low` it's fully on (maximum correction), above `high` it's fully off (no correction), and in between it fades gradually from one to the other. The model sees a perfectly continuous function of dimension index.

> **Why does smoothness matter?** If we applied full correction below `low` and zero correction above `high` with a hard cutoff, adjacent frequency pairs would be treated very differently at the boundary. That discontinuity in position encoding would confuse the model. The ramp keeps things gradual, the way a good equalizer adjusts audio frequencies without harsh notches.

In [ ]:
def linear_ramp_factor(min_val, max_val, n):
    if min_val == max_val:
        max_val += 0.001
    return [min(1.0, max(0.0, (i - min_val) / (max_val - min_val))) for i in range(n)]

ramp = linear_ramp_factor(low, high, d // 2)
smooth = [1.0 - r for r in ramp]

print(f"Ramp and smooth factors (showing key pairs around [{low}, {high}]):")
print(f"{'pair i':>8} {'ramp(i)':>10} {'smooth(i)':>11} {'region':>20}")
for i in range(d // 2):
    region = "no-scale" if i < low else ("full-scale" if i > high else "blend")
    if i <= low + 2 or i >= high - 2 or region == "blend":
        print(f"{i:8d} {ramp[i]:10.4f} {smooth[i]:11.4f} {region:>20}")


## The YaRN Formula

Everything comes together in one clean expression. For each frequency pair $i$, the modified frequency $\tilde{\theta}_i$ blends two extreme treatments:

$$\boxed{\tilde{\theta}_i = \frac{\theta_i}{\text{factor}} \cdot (1 - \text{smooth}_i) + \theta_i \cdot \text{smooth}_i}$$

Let's unpack the two terms side by side:

| Term | When it dominates | What it does |
|------|------------------|--------------|
| $\theta_i / \text{factor}$ | $\text{smooth}_i \approx 0$ (low-freq pairs) | Slows the pair by `factor` — identical to linear position interpolation |
| $\theta_i$ | $\text{smooth}_i \approx 1$ (high-freq pairs) | Leaves the pair completely unchanged |

The blending coefficient $\text{smooth}_i$ makes the transition continuous — at the boundary between slow and fast, both terms contribute equally.

> **Sanity check:** With `factor = 40` for a 40× context extension, the slowest pairs rotate 40× more slowly than before. A pair that previously completed one cycle over 4,096 tokens now takes 163,840 tokens — perfectly calibrated for the extended context length.

In [ ]:
factor = 40.0  # rope_factor from DSpark config

# Apply YaRN modification
freqs_yarn = [
    freqs_base[i] / factor * (1.0 - smooth[i]) + freqs_base[i] * smooth[i]
    for i in range(d // 2)
]

print(f"Frequency modification (factor={factor}, low={low}, high={high}):")
print(f"{'pair i':>8} {'theta_base':>12} {'theta_yarn':>12} {'ratio':>8} {'change'}")
for i in [0, 4, 8, low, (low+high)//2, high, high+2, 31]:
    ratio = freqs_yarn[i] / freqs_base[i]
    change = "unchanged" if abs(ratio - 1.0) < 0.01 else (f"/={1/ratio:.1f}" if ratio < 1 else f"*={ratio:.2f}")
    print(f"{i:8d} {freqs_base[i]:12.6f} {freqs_yarn[i]:12.6f} {ratio:8.4f} {change}")

# Verify: full-scale region approaches 1/factor
print(f"\nExpected ratio for fully-scaled pairs: 1/factor = {1/factor:.4f}")
print(f"Actual ratio at pair {d//2-1}: {freqs_yarn[d//2-1]/freqs_base[d//2-1]:.4f}")


## Precomputing the Frequency Table

With the modified frequencies $\tilde{\theta}_i$ in hand, we precompute a lookup table of complex exponentials — one entry per (position $t$, frequency pair $i$):

$$\text{freqs\_cis}[t, i] = e^{j \cdot t \cdot \tilde{\theta}_i} = \cos(t \cdot \tilde{\theta}_i) + j \cdot \sin(t \cdot \tilde{\theta}_i)$$

This step is identical to standard RoPE — we're just plugging in YaRN-adjusted frequencies instead of the original ones. The table is computed once at model load time and re-used at every single forward pass, so it has zero runtime overhead.

> **Memory note:** With `d_rope = 64` and `max_seq_len = 4096`, the table holds $4096 \times 32$ complex numbers — roughly 0.5 MB. You won't even notice it.

In [ ]:
def precompute_freqs_cis(dim, seqlen, original_seq_len, base, factor, beta_fast, beta_slow):
    """Precomputes (cos, sin) table for YaRN-modified RoPE."""
    n_pairs = dim // 2
    # Step 1: baseline frequencies
    freqs = [1.0 / (base ** (2 * i / dim)) for i in range(n_pairs)]
    # Step 2: YaRN modification
    if original_seq_len > 0:
        lo, hi = find_correction_range(beta_fast, beta_slow, dim, base, original_seq_len)
        ramp = linear_ramp_factor(lo, hi, n_pairs)
        smooth = [1.0 - r for r in ramp]
        freqs = [f / factor * (1 - s) + f * s for f, s in zip(freqs, smooth)]
    # Step 3: outer product -> (seqlen, n_pairs) table
    # freqs_cis[t][i] = (cos(t*theta_i), sin(t*theta_i))
    freqs_cis = []
    for t in range(seqlen):
        row = [(math.cos(t * theta), math.sin(t * theta)) for theta in freqs]
        freqs_cis.append(row)
    return freqs_cis

# DSpark-like config (small for demonstration)
freqs_cis = precompute_freqs_cis(
    dim=64, seqlen=10, original_seq_len=4096,
    base=10000.0, factor=40.0, beta_fast=32, beta_slow=1
)

print("freqs_cis[t][i] = (cos(t*theta_i), sin(t*theta_i)):")
print(f"{'t':>4} {'pair 0 cos':>12} {'pair 0 sin':>12} {'pair 31 cos':>13} {'pair 31 sin':>13}")
for t in range(6):
    c0, s0 = freqs_cis[t][0]
    c31, s31 = freqs_cis[t][31]
    print(f"{t:4d} {c0:12.6f} {s0:12.6f} {c31:13.6f} {s31:13.6f}")
print("\nPair 0 rotates fast (high freq), pair 31 is nearly constant (slowed by YaRN).")


## How `apply_rotary_emb` Uses the Table

`apply_rotary_emb` treats adjacent pairs of dimensions as the real and imaginary parts of a complex number and multiplies each pair by the corresponding entry in the precomputed table. For a head vector $[x_0, x_1, x_2, x_3, \ldots]$, the pairs $(x_0, x_1)$, $(x_2, x_3)$, and so on each get rotated by their own angle. In matrix form, pair $i$ at position $t$ applies:

$$R_i(t) = \begin{pmatrix} \cos(t\tilde{\theta}_i) & -\sin(t\tilde{\theta}_i) \\ \sin(t\tilde{\theta}_i) & \cos(t\tilde{\theta}_i) \end{pmatrix}$$

After applying these rotations to both Q and K, the dot product Q · K naturally encodes only the *relative distance* $t_Q - t_K$ between the two tokens — that's the core promise of RoPE, now extended to long contexts without modification.

> **Inverse mode.** DSpark calls `apply_rotary_emb(o, freqs_cis, conjugate=True)` on the output after attention. This *undoes* the rotation, leaving the output vectors in a position-independent space — useful when subsequent layers don't need positional encoding.

## RTX 4080: YaRN Adds Zero Runtime Cost

YaRN's work happens entirely at **model load time**, not during inference.

**At startup (once):**
1. Compute the correction range from `beta_fast` and `beta_slow` — a tiny loop over d/2 pairs
2. Build the smooth ramp — another loop over d/2 values
3. Apply the frequency modification — one expression per pair
4. Precompute the full `(cos, sin)` table for every position and every pair

**At inference (every token):**
Just look up `freqs_cis[seq_pos, pair]` — the table is already computed and cached.

**Table size for Qwen3-8B:** `rope_head_dim=64`, max `seq_len=4096`.
`4096 positions × 32 pairs × 2 (cos + sin) × 4 bytes = 1 MB`.

The RTX 4080 has 64 MB of L2 cache. After the first forward pass, the table lives in L2
and every subsequent token reads from it at ~7 TB/s — essentially free.

**YaRN vs standard RoPE kernel: identical.**
The only difference is the values baked into `freqs_cis`.
No new PTX instructions. No different memory pattern. No extra math per token.

**Context length extension to 32K:**
```
factor = 32768 / 4096 = 8
freqs_cis table grows: 4096 × 32 → 32768 × 32 entries
Table size: 1 MB → 8 MB  (still fits in L2 with room to spare)
```
Larger table, same lookup, same kernel.

## CuTeDSL Connection: apply_rotary_emb

`apply_rotary_emb` applies the precomputed freq table to a batch of head vectors. It's structurally identical to the standard RoPE kernel — the YaRN-modified frequencies are already baked into `freqs_cis` before the kernel runs.

```python
@cute.kernel
def apply_rotary_emb_kernel(mX, mFreqCos, mFreqSin, mOut, conjugate=False):
    # One thread per (seq_pos, head, freq_pair)
    seq_pos = blockIdx.x
    head    = blockIdx.y
    pair    = threadIdx.x   # 0 to rope_head_dim//2 - 1  (= 31 for Qwen3)

    x0 = mX[seq_pos, head, 2*pair]
    x1 = mX[seq_pos, head, 2*pair + 1]

    cos_m = mFreqCos[seq_pos, pair]   # precomputed by precompute_freqs_cis()
    sin_m = mFreqSin[seq_pos, pair]   # YaRN-adjusted frequency already applied

    sign = -1.0 if conjugate else 1.0   # conjugate=True undoes the rotation

    mOut[seq_pos, head, 2*pair]     = x0 * cos_m - sign * x1 * sin_m
    mOut[seq_pos, head, 2*pair + 1] = sign * x0 * sin_m + x1 * cos_m

    # NoPE dims (pair >= rope_head_dim//2): handled in a separate copy loop,
    # or skipped if mX and mOut are the same buffer (in-place).
```

**Conjugate mode** is used for the output projection: after computing the attention output, the rotation applied to Q and K is "undone" on the output vectors to return to a position-independent representation. This is a Qwen3 design choice — not all models do this.

## Where YaRN Fits in the Qwen3-8B Extended-Context Decode Step

YaRN adds no new runtime operation — it changes the *values* baked into the precomputed frequency table. The inference code path is identical to standard RoPE after model load.

```
Model load time (once per session):
  ① Compute correction range [low, high] from beta_fast and beta_slow
  ② Build smooth ramp and apply per-pair frequency rescaling
  ③ Precompute freq table:
       freqs_cos[pos, pair] = cos(pos × theta_yarn[pair])
       freqs_sin[pos, pair] = sin(pos × theta_yarn[pair])
     Table shape: [max_seq_len, rope_head_dim//2] = [32768, 32]
     Table size:  32768 × 32 × 2 × 4 bytes = 8 MB

Each decode step (per token generated):
  ④ RoPE on Q at position t: look up freqs_cos[t, :], freqs_sin[t, :]
  ⑤ RoPE on K at position t: same lookup, same kernel — no YaRN-specific code
```

### What changes at different extension factors

| Target context | factor | Freq table size | Slowest pair wavelength |
|----------------|--------|-----------------|------------------------|
| 4,096 (base)   | 1      | 1 MB            | 10,000 tokens          |
| 32,768 (8×)    | 8      | 8 MB            | 80,000 tokens          |
| 131,072 (32×)  | 32     | 32 MB           | 320,000 tokens         |

At 32× extension, the freq table is 32 MB — still fits in L2 (64 MB on RTX 4080).

### Why the model does not need retraining

The YaRN correction ensures that even at position 131,072, each frequency pair applies a rotation within the angle range it saw during training:
- High-freq pairs (fully trained at all angles): left unchanged
- Low-freq pairs (barely rotate within training window): slowed by `factor` so they reach the same max angle at 131K that they reached at 4096 during training

The dot-product `Q[t] · K[s]` still evaluates to a familiar rotation difference — the model's attention patterns generalize correctly.

## YaRN in the Wild

### DeepSeek-V4 / DSpark

YaRN is the positional encoding used by DeepSeek models with extended context.
The config parameters `rope_factor`, `original_seq_len`, `beta_fast`, `beta_slow`
are stored in `config.json` and loaded by `model.py::precompute_freqs_cis`.
For DSpark, `original_seq_len = 4096`, `rope_factor = 40`, targeting a 163,840-token context.

### HuggingFace Transformers

`transformers/modeling_rope_utils.py::_compute_yarn_parameters` — the reference implementation.
Loaded automatically when `rope_scaling.type = "yarn"` is set in `config.json`.
Used by Mistral, LLaMA-3 long context, and DeepSeek checkpoints on the Hub.

### vLLM

`vllm/model_executor/layers/rotary_embedding.py::YaRNScalingRotaryEmbedding`
Computes the freq table at init using the same math as HuggingFace.
The runtime path is identical to standard RoPE — the extended table is indexed by `position_ids`.

### LongRoPE and LongLLaMA (related methods)

LongRoPE (Microsoft) uses a non-uniform per-dimension rescaling rather than the smooth ramp.
LongLLaMA uses simple linear interpolation without a correction range.
Both are structurally identical at runtime: modify freq table at load time, run standard RoPE at inference.

### Pattern shared across all YaRN implementations

1. Read `rope_factor`, `original_seq_len`, `beta_fast`, `beta_slow` from config
2. Compute correction range `[low, high]` using `find_correction_range`
3. Build smooth ramp and apply per-pair rescaling
4. Precompute full `(cos, sin)` table up to `max_position_embeddings`
5. At inference: **identical to standard RoPE** — just different values in the table

Zero runtime overhead. All complexity is in the one-time table construction.

## One-Sentence Takeaway

YaRN extends a model's useful context length by 8–40× without retraining by identifying exactly which rotary frequency pairs would visit out-of-distribution angles at long sequences and slowing precisely those pairs down via a smooth linear ramp — the entire trick is in a one-time frequency table computed at model load time, after which inference is indistinguishable from standard RoPE, with the only runtime cost being a slightly larger precomputed table (8–32 MB) that easily fits in the GPU's L2 cache even at 32K context.

## Community Optimization Resources for YaRN RoPE on SM89

### YaRN adds zero runtime cost — all work is at model load time

YaRN is architecturally identical to standard RoPE at inference. The only difference is the precomputed frequency table. All community benchmarks for RoPE optimization apply equally to YaRN.

| Stage | Work done | Runtime cost |
|---|---|---|
| Model load (once) | Compute correction range, build smooth ramp, precompute (cos, sin) table | ~1 ms total |
| Each decode token | Look up `freq_cos[t, pair]`, `freq_sin[t, pair]` from table | Zero vs standard RoPE |
| Each prefill pass | Same lookup, same kernel | Zero vs standard RoPE |

---

### Frequency table sizing for Qwen3-8B at different context lengths

Qwen3-8B uses `rope_head_dim=64` → 32 frequency pairs, `rope_theta=1,000,000`.

| Context length | Extension factor | Table size | Fits in L2 (64 MB)? |
|---|---|---|---|
| 4,096 (base) | 1× | 4096 × 32 × 2 × 4 B = 1 MB | ✅ Always cached |
| 32,768 (8×) | 8× | 8 MB | ✅ Fits with room to spare |
| 131,072 (32×) | 32× | 32 MB | ✅ Fits in 64 MB L2 |
| 524,288 (128×) | 128× | 128 MB | ❌ Exceeds L2 — table reloaded from HBM |

For Qwen3-8B's default `max_position_embeddings=32768`, the YaRN table is 8 MB — the RTX 4080 L2 can hold it permanently after the first token.

---

### What to optimize for YaRN specifically

**Nothing at the kernel level.** The kernel is the same `apply_rotary_emb` used for standard RoPE. CuTeDSL improvements to the base RoPE kernel automatically apply to YaRN.

The one YaRN-specific optimization target is the **frequency table precomputation** itself:

```python
# Current Python implementation: O(max_seq_len × n_pairs) loop
# For max_seq_len=32768, n_pairs=32: 1,048,576 iterations
# Time: ~10 ms in Python, ~0.1 ms if vectorized with torch or numpy

# Better: vectorize with torch
positions = torch.arange(max_seq_len)
freqs = torch.tensor(freqs_yarn)  # [n_pairs] — already computed
# outer product:
angles = positions.unsqueeze(1) * freqs.unsqueeze(0)  # [max_seq_len, n_pairs]
freq_cos = angles.cos()
freq_sin = angles.sin()
# ~ 5-50× faster than the Python loop
```

---

### Community references for long-context RoPE extensions

| Method | Where used | Key difference from YaRN |
|---|---|---|
| **YaRN** | DeepSeek, Mistral, some Qwen variants | Smooth frequency ramp + attention scaling |
| **LongRoPE** | Microsoft Phi-3 long context | Per-dimension non-uniform rescaling (no smooth ramp) |
| **Dynamic NTK** | Community, early long-context LLaMA | Adjusts `base` dynamically per sequence length |
| **Linear interpolation** | Position-interpolation (PI, 2023) | Uniform scaling of all frequencies — YaRN is strictly better |

**HuggingFace reference:** `transformers/modeling_rope_utils.py::_compute_yarn_parameters` — the canonical Python implementation used by all HF YaRN models including DeepSeek checkpoints.

**vLLM reference:** `vllm/model_executor/layers/rotary_embedding.py::YaRNScalingRotaryEmbedding` — same math, runs at init, same runtime kernel as standard RoPE.